# m_to_case_helper: MATPOWER `.m` → GridKit `case.json`

* conda: `h-py312-basic`
* notes and design details: [`work-notes/m_to_case_helper.md`](../work-notes/m_to_case_helper.md)

Isolated validation notebook for the `.m` → `case.json` mapping. This is the
pipeline stage that sits between the PF solve (see [`pm_helper.ipynb`](pm_helper.ipynb))
and the dynamic simulation (see `aleatoric_helper.ipynb`, to come).

## Two coexisting entry points

| Function | Use when | Modifies |
|---|---|---|
| `patch_case_from_m()` | Only `PG`, `QG`, `Vm`, `Va` changed | bus `init.Vr/Vi`, Genrou `p0/q0` |
| `build_case_from_solved_m()` | Load, wind, or generator commitment changed | Same as above **plus** `LoadZIP.{Pnom,Qnom,Vnom}`, plus removal of `{Genrou, Tgov1, SexsPti/Ieeet1}` triplets for offline gens |

`patch_case_from_m` is unchanged and remains the default for the epistemic
track (`uq_setup.ipynb`, illinois-v2). The new `build_case_from_solved_m` is
the aleatoric-track workhorse.

## Workflow of this notebook

1. **Section 1**: imports and path setup
2. **Section 2**: quick look at the base `illinois.json` structure (device
   classes, counts, LoadZIP layout)
3. **Section 3**: identity round-trip — PM.jl-solve the base `.m`, feed to
   `build_case_from_solved_m`, structural + numerical compare vs `illinois.json`
4. **Section 4**: field-by-field unit tests
    - 4a. bus `init.Vr/Vi` from a single-bus `Vm/Va` edit (no PF solve)
    - 4b. Genrou `p0/q0` from a single-gen `PG` edit
    - 4c. offline-gen removal (device triplet drop)
    - 4d. LoadZIP single-bus `Pnom/Qnom/Vnom` patch
    - 4e. LoadZIP multi-ZIP proportional-scaling exactness
    - 4f. shunt / capacitor exclusion (bus 15)
    - 4g. zero-load bus handling
5. **Section 5**: end-to-end with a real perturbation (0.8x load scale +
   PM.jl solve + build case.json). Sanity-check aggregate PD, PG conservation.
6. **Section 6**: findings summary



# Section 1: imports and path setup


In [ ]:
import copy
import importlib
import json
import math
import os
import shutil
import subprocess
import sys
import tempfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.6f}".format)

# === repo layout ===
GRIDKIT_REPO = Path.home() / "gridkit"
UQ_DIR = GRIDKIT_REPO / "uq-usecase"
PY_UTILS_DIR = UQ_DIR / "py-utils"

# === PM.jl driver ===
PM_PROJECT_DIR = UQ_DIR / "pm-solver"
PM_SOLVE_JL = PM_PROJECT_DIR / "pm_solve.jl"
JULIA_BIN = Path.home() / "bin/julia112"

# === base MATPOWER case + base GridKit JSON ===
BASECASE_M = Path(
    "/kfs2/projects/scidac/scidac-data/ACTIVSg200/raw-tamu-data/case_ACTIVSg200.m"
)
ILLINOIS_JSON = (
    GRIDKIT_REPO / "build/examples/PhasorDynamics/Large/Illinois/illinois.json"
)

# === work dir for this helper (all temporary / test artifacts) ===
WORK_DIR = UQ_DIR / "notebooks/test-runs/m_to_case_helper"
WORK_DIR.mkdir(parents=True, exist_ok=True)

# === py-utils on path ===
if str(PY_UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(PY_UTILS_DIR))

import m_to_case
import pf_utils
import pm_utils

importlib.reload(m_to_case)
importlib.reload(pf_utils)
importlib.reload(pm_utils)

from m_to_case import (
    build_case_from_solved_m,
    build_cases_from_solved_m_list,
    patch_case_from_m,
)

for label, p in [
    ("BASECASE_M", BASECASE_M),
    ("ILLINOIS_JSON", ILLINOIS_JSON),
    ("PM_SOLVE_JL", PM_SOLVE_JL),
    ("JULIA_BIN", JULIA_BIN),
    ("WORK_DIR", WORK_DIR),
]:
    status = "OK " if p.exists() else "MISSING"
    print(f"  [{status}]  {label}: {p}")

# Section 2: quick look at base `illinois.json` structure

Establish baseline counts for what we're patching. These numbers feed directly
into the expected-count assertions in later sections (Section 3 identity
round-trip, Section 4 per-family tests).


In [ ]:
with open(ILLINOIS_JSON) as fh:
    ILLINOIS = json.load(fh)

n_buses_json = len(ILLINOIS.get("buses", []))
device_class_counts = Counter(d.get("class") for d in ILLINOIS.get("devices", []))

print(f"illinois.json bus count       : {n_buses_json}")
print(f"illinois.json device classes  :")
for cls, cnt in sorted(device_class_counts.items(), key=lambda kv: -kv[1]):
    print(f"  {cls:<20s} {cnt}")

In [ ]:
# LoadZIP grouping: how many devices per bus, how many shunt-like devices.
loadzip_by_bus = {}
for dev in ILLINOIS.get("devices", []):
    if dev.get("class") != "LoadZIP":
        continue
    bus = dev.get("ports", {}).get("bus")
    loadzip_by_bus.setdefault(bus, []).append(dev)

n_load_buses = len(loadzip_by_bus)
multi_zip_buses = [b for b, devs in loadzip_by_bus.items() if len(devs) > 1]

# Shunt / capacitor: Pnom==0 AND Qnom<0 OR id starts with "shunt_".
shunt_like = []
for bus, devs in loadzip_by_bus.items():
    for d in devs:
        p = d.get("params", {})
        pnom = float(p.get("Pnom", 0.0))
        qnom = float(p.get("Qnom", 0.0))
        did = str(d.get("id", ""))
        if did.startswith("shunt_") or (pnom == 0.0 and qnom < 0.0):
            shunt_like.append((bus, d.get("id"), pnom, qnom))

print(f"LoadZIP buses           : {n_load_buses}")
print(f"multi-ZIP buses (>=2)   : {len(multi_zip_buses)}")
print(f"shunt-like LoadZIP devs : {len(shunt_like)}")
for b, i, p, q in shunt_like:
    print(f"  bus {b}: id={i}, Pnom={p:.6f}, Qnom={q:.6f}")

genrou_ids = [
    d.get("id") for d in ILLINOIS.get("devices", []) if d.get("class") == "Genrou"
]
print(f"Genrou devices          : {len(genrou_ids)}")
print(f"  first 5 ids           : {genrou_ids[:5]}")

In [ ]:
# Companion-device counts: how many Tgov1 / SexsPti / Ieeet1 devices, for
# expected-remove-count arithmetic when we drop offline gens.
companion_classes = ["Genrou", "Tgov1", "SexsPti", "Ieeet1"]
companion_counts = {c: device_class_counts.get(c, 0) for c in companion_classes}
companion_counts

# Section 3: identity round-trip

PM.jl-solve the base `.m`; feed the solved `.m` into
`build_case_from_solved_m(illinois.json, base_solved.m, out.json)`; compare
the resulting JSON against `illinois.json`.

Structural expectations (must hold exactly):
- Same set of device classes.
- Same set of Genrou / LoadZIP / Tgov1 / SexsPti ids (no additions, no drops
  in this case since the base .m has nothing offline that's online in the JSON).
- Same bus `number`s.

Numerical expectations (within tolerance):
- `bus.init.Vr/Vi` differ by whatever PM.jl vs the base JSON's original solver
  disagree on (see `pm_helper.md` cross-comparison; typical `dVm` order 1e-3
  to 1e-2 pu).
- `Genrou.p0/q0` differ by the same solver-equilibrium residual on PG/QG.
- `LoadZIP.Pnom/Qnom` **will** change vs the base JSON: per `illinois.md`,
  base JSON `Pnom` is systematically ~1.49x higher than `PD/100`. The
  invariant we check is that the multi-ZIP proportion between devices at the
  same bus is preserved.

See [`work-notes/m_to_case_helper.md`](../work-notes/m_to_case_helper.md) §
"The identity round-trip test" for the full rationale.


In [ ]:
# 3.1 PM.jl-solve the base case.
BASE_SOLVED_M = WORK_DIR / "case_ACTIVSg200_pm_solved.m"

df_pm, stderr, rc = pm_utils.run_pm_solve_out(
    pm_project_dir=PM_PROJECT_DIR,
    m_path=BASECASE_M,
    out_m_path=BASE_SOLVED_M,
    julia_bin=JULIA_BIN,
    pm_solve_jl=PM_SOLVE_JL,
)
pm_utils.pm_summary("base_ACTIVSg200 (PM.jl warm)", df_pm, rc, stderr)
BASE_SOLVED_M.exists(), BASE_SOLVED_M.stat().st_size

In [ ]:
# 3.2 Build case.json from the PM.jl-solved base .m.
OUT_JSON = WORK_DIR / "illinois_from_base_pm_solved.json"

case_out, counts = build_case_from_solved_m(
    base_case_json_path=str(ILLINOIS_JSON),
    m_path=str(BASE_SOLVED_M),
    output_path=str(OUT_JSON),
    patch_bus_init=True,
    patch_gen_dispatch=True,
    patch_load=True,
    remove_offline_gens=True,
    strict=True,
    verbose=True,
)
counts

In [ ]:
# 3.3 Structural check: same bus set, same device class counts.
class_counts_out = Counter(d.get("class") for d in case_out.get("devices", []))

structural_report = pd.DataFrame(
    [
        (
            "bus count",
            len(ILLINOIS.get("buses", [])),
            len(case_out.get("buses", [])),
        ),
    ]
    + [
        (f"class:{c}", device_class_counts.get(c, 0), class_counts_out.get(c, 0))
        for c in sorted(set(device_class_counts) | set(class_counts_out))
    ],
    columns=["field", "base_json", "built_json"],
)
structural_report["match"] = (
    structural_report["base_json"] == structural_report["built_json"]
)
structural_report

In [ ]:
# 3.4 Numerical residuals on bus Vr / Vi.
def bus_frame(case_dict):
    rows = []
    for b in case_dict.get("buses", []):
        init = b.get("init", {})
        rows.append(
            {
                "number": b.get("number"),
                "Vr": init.get("Vr"),
                "Vi": init.get("Vi"),
            }
        )
    df = pd.DataFrame(rows).set_index("number").sort_index()
    df["Vm"] = np.sqrt(df["Vr"] ** 2 + df["Vi"] ** 2)
    df["Va_deg"] = np.degrees(np.arctan2(df["Vi"], df["Vr"]))
    return df


bus_base = bus_frame(ILLINOIS)
bus_out = bus_frame(case_out)
bus_delta = pd.DataFrame(
    {
        "Vm_base": bus_base["Vm"],
        "Vm_out": bus_out["Vm"],
        "dVm": bus_out["Vm"] - bus_base["Vm"],
        "Va_base": bus_base["Va_deg"],
        "Va_out": bus_out["Va_deg"],
        "dVa_deg": bus_out["Va_deg"] - bus_base["Va_deg"],
    }
)
print("bus Vm / Va residuals (out - base):")
print(bus_delta[["dVm", "dVa_deg"]].describe(percentiles=[0.5, 0.95, 0.99]))
print()
print(f"max |dVm|      : {bus_delta['dVm'].abs().max():.6f} pu")
print(f"max |dVa_deg|  : {bus_delta['dVa_deg'].abs().max():.6f} deg")

In [ ]:
# 3.5 Genrou p0 / q0 residuals.
def genrou_frame(case_dict):
    rows = []
    for d in case_dict.get("devices", []):
        if d.get("class") != "Genrou":
            continue
        p = d.get("params", {})
        rows.append({"id": d.get("id"), "p0": p.get("p0"), "q0": p.get("q0")})
    return pd.DataFrame(rows).set_index("id").sort_index()


gen_base = genrou_frame(ILLINOIS)
gen_out = genrou_frame(case_out)
gen_delta = pd.DataFrame(
    {
        "p0_base": gen_base["p0"],
        "p0_out": gen_out["p0"],
        "dp0": gen_out["p0"] - gen_base["p0"],
        "q0_base": gen_base["q0"],
        "q0_out": gen_out["q0"],
        "dq0": gen_out["q0"] - gen_base["q0"],
    }
)
gen_delta.describe(percentiles=[0.5, 0.95, 0.99])

In [ ]:
# 3.6 LoadZIP invariant: multi-ZIP proportions preserved.
def loadzip_frame(case_dict):
    rows = []
    for d in case_dict.get("devices", []):
        if d.get("class") != "LoadZIP":
            continue
        p = d.get("params", {})
        rows.append(
            {
                "id": d.get("id"),
                "bus": d.get("ports", {}).get("bus"),
                "Pnom": float(p.get("Pnom", 0.0)),
                "Qnom": float(p.get("Qnom", 0.0)),
                "Vnom": float(p.get("Vnom", 0.0)),
            }
        )
    return pd.DataFrame(rows).set_index("id").sort_index()


lz_base = loadzip_frame(ILLINOIS)
lz_out = loadzip_frame(case_out)


# Per-bus multi-ZIP: within each bus, compute Pnom fractions in base vs out.
# For non-shunt real loads, fractions must be preserved to floating-point.
def per_bus_fractions(df):
    df = df.copy()
    df["is_shunt"] = (df["Pnom"] == 0.0) & (df["Qnom"] < 0.0)
    real = df[~df["is_shunt"]].copy()
    real["bus_sum_P"] = real.groupby("bus")["Pnom"].transform("sum")
    real["P_frac"] = np.where(
        real["bus_sum_P"] > 0, real["Pnom"] / real["bus_sum_P"], np.nan
    )
    return real


frac_base = per_bus_fractions(lz_base)[["bus", "P_frac"]].rename(
    columns={"P_frac": "P_frac_base"}
)
frac_out = per_bus_fractions(lz_out)[["bus", "P_frac"]].rename(
    columns={"P_frac": "P_frac_out"}
)
frac_join = frac_base.join(frac_out[["P_frac_out"]])
frac_join["d_frac"] = frac_join["P_frac_out"] - frac_join["P_frac_base"]

# Only multi-ZIP buses are meaningful for this test (N=1 buses have P_frac=1
# by definition and would give d_frac=0 trivially).
multi = frac_join.groupby("bus").filter(lambda g: len(g) > 1)
if len(multi):
    print(
        f"multi-ZIP fraction invariant across {multi['bus'].nunique()} buses / "
        f"{len(multi)} devices:"
    )
    print(f"  max |d_frac| : {multi['d_frac'].abs().max():.3e}")
    print(f"  mean|d_frac| : {multi['d_frac'].abs().mean():.3e}")
else:
    print("(no multi-ZIP buses found — check illinois.md ~ 38 expected)")

In [ ]:
# 3.7 LoadZIP aggregate: sum(Pnom) per bus should equal PD/baseMVA from the
# solved .m for every non-shunt bus.
from matpowercaseframes import Case as MPCase

mp = MPCase(str(BASE_SOLVED_M))
baseMVA = float(mp.baseMVA)
bus_df = mp.bus
bus_pd = bus_df.set_index("BUS_I")[["PD", "QD"]].astype(float)
bus_pd["Pnom_pu_from_m"] = bus_pd["PD"] / baseMVA
bus_pd["Qnom_pu_from_m"] = bus_pd["QD"] / baseMVA

# Aggregate output LoadZIP by bus, excluding shunt-like.
lz_out_real = lz_out[~((lz_out["Pnom"] == 0.0) & (lz_out["Qnom"] < 0.0))]
agg_out = lz_out_real.groupby("bus")[["Pnom", "Qnom"]].sum()
agg_out.columns = ["Pnom_out_sum", "Qnom_out_sum"]

check = agg_out.join(bus_pd[["Pnom_pu_from_m", "Qnom_pu_from_m"]], how="inner")
check["dPnom"] = check["Pnom_out_sum"] - check["Pnom_pu_from_m"]
check["dQnom"] = check["Qnom_out_sum"] - check["Qnom_pu_from_m"]
print(f"aggregate LoadZIP vs solved .m PD/QD across {len(check)} buses:")
print(f"  max |dPnom| : {check['dPnom'].abs().max():.3e} pu")
print(f"  max |dQnom| : {check['dQnom'].abs().max():.3e} pu")

# Section 4: field-by-field unit tests

Each test edits a single field in a copy of the base `.m` (no PF solve),
runs `build_case_from_solved_m` with the right toggles, and verifies that
only the expected fields changed. This isolates each mapping rule so a
regression here points at a specific code path.

Helper: `_edit_m_bus_col`, `_edit_m_gen_col` rewrite a single `mpc.bus` or
`mpc.gen` column value at a single row, preserving all other bytes.


In [ ]:
# Helpers: single-cell edits to mpc.bus / mpc.gen in a .m file, preserving
# the rest byte-for-byte. Uses pf_utils._parse_block / _replace_block, the
# same low-level parser already used by make_perturbed_load_m etc.
from pf_utils import _parse_block, _replace_block


def _edit_m_bus_col(src_m, dst_m, bus_i, col_1based, new_value):
    """Rewrite mpc.bus[row where BUS_I==bus_i, col_1based] = new_value."""
    src_m = Path(src_m)
    dst_m = Path(dst_m)
    dst_m.parent.mkdir(parents=True, exist_ok=True)
    content = src_m.read_text()
    lines = _parse_block(content, "bus")
    new_lines = []
    found = False
    for line in lines:
        parts = line.strip().rstrip(";").split()
        if int(float(parts[0])) == int(bus_i):
            parts[col_1based - 1] = f"{new_value:.6f}"
            found = True
            new_lines.append("\t" + "\t".join(parts) + ";")
        else:
            new_lines.append(line)
    if not found:
        raise ValueError(f"BUS_I={bus_i} not found in {src_m}")
    new_block = "\n".join(new_lines)
    new_content = _replace_block(content, "bus", new_block)
    dst_m.write_text(new_content)
    return dst_m


def _edit_m_gen_col(src_m, dst_m, gen_bus, rank, col_1based, new_value):
    """Rewrite mpc.gen[rank-th row at bus=gen_bus, col_1based] = new_value."""
    src_m = Path(src_m)
    dst_m = Path(dst_m)
    dst_m.parent.mkdir(parents=True, exist_ok=True)
    content = src_m.read_text()
    lines = _parse_block(content, "gen")
    new_lines = []
    seen_at_bus = 0
    found = False
    for line in lines:
        parts = line.strip().rstrip(";").split()
        if int(float(parts[0])) == int(gen_bus):
            seen_at_bus += 1
            if seen_at_bus == rank:
                parts[col_1based - 1] = f"{new_value:.6f}"
                found = True
                new_lines.append("\t" + "\t".join(parts) + ";")
                continue
        new_lines.append(line)
    if not found:
        raise ValueError(f"gen at bus={gen_bus}, rank={rank} not found in {src_m}")
    new_block = "\n".join(new_lines)
    new_content = _replace_block(content, "gen", new_block)
    dst_m.write_text(new_content)
    return dst_m


# Quick smoke test on the helpers: make a no-op edit and confirm .m still parses.
_probe = _edit_m_bus_col(
    BASECASE_M, WORK_DIR / "_probe.m", bus_i=2, col_1based=8, new_value=1.0
)
probe_case = MPCase(str(_probe))
print("probe bus edit OK; bus count =", len(probe_case.bus))

## 4a. bus init only

Edit `mpc.bus` `VM` at one bus by +0.05 pu (leave `VA` unchanged). Run
`build_case_from_solved_m` with `patch_gen_dispatch=False`, `patch_load=False`,
`remove_offline_gens=False`. Verify that only the target bus's `Vr, Vi`
changed and by the exact trig-derived value.


In [ ]:
TARGET_BUS_4A = 2
# read base VM/VA at that bus (from BASE_SOLVED_M to keep everything else consistent)
base_mp = MPCase(str(BASE_SOLVED_M))
base_row = base_mp.bus.set_index("BUS_I").loc[TARGET_BUS_4A]
vm_base, va_base = float(base_row["VM"]), float(base_row["VA"])
vm_new = vm_base + 0.05
print(
    f"bus {TARGET_BUS_4A}: VM {vm_base:.6f} -> {vm_new:.6f}, VA {va_base:.6f} (unchanged)"
)

M_4A = _edit_m_bus_col(
    BASE_SOLVED_M,
    WORK_DIR / "4a_bus_init.m",
    bus_i=TARGET_BUS_4A,
    col_1based=8,
    new_value=vm_new,
)

case_4a, counts_4a = build_case_from_solved_m(
    base_case_json_path=str(ILLINOIS_JSON),
    m_path=str(M_4A),
    output_path=None,
    patch_bus_init=True,
    patch_gen_dispatch=False,
    patch_load=False,
    remove_offline_gens=False,
    strict=False,
    verbose=False,
)

# Compute expected Vr, Vi.
va_rad = math.radians(va_base)
vr_expected = vm_new * math.cos(va_rad)
vi_expected = vm_new * math.sin(va_rad)

# Locate the target bus in the output.
target = next(b for b in case_4a["buses"] if b.get("number") == TARGET_BUS_4A)
vr_got = target["init"]["Vr"]
vi_got = target["init"]["Vi"]
print(f"expected Vr={vr_expected:.9f}, Vi={vi_expected:.9f}")
print(f"got      Vr={vr_got:.9f}, Vi={vi_got:.9f}")
print(
    f"|dVr| = {abs(vr_got - vr_expected):.3e}, |dVi| = {abs(vi_got - vi_expected):.3e}"
)

# Confirm no other bus changed (compare against the identity round-trip output).
n_other_changed = 0
for b_out, b_ref in zip(case_4a["buses"], case_out["buses"]):
    if b_out.get("number") == TARGET_BUS_4A:
        continue
    if (
        abs(b_out["init"]["Vr"] - b_ref["init"]["Vr"]) > 1e-12
        or abs(b_out["init"]["Vi"] - b_ref["init"]["Vi"]) > 1e-12
    ):
        n_other_changed += 1
print(f"other buses changed: {n_other_changed} (expected 0)")

## 4b. Genrou dispatch only

Edit `PG` for one online generator by +100 MW; check that the corresponding
Genrou `p0` in the output changed by exactly `100 / baseMVA`.


In [ ]:
# Pick the first online gen in illinois.json that has PG > 0 in the base .m.
gen_df = base_mp.gen.copy()
gen_df["rank_at_bus"] = gen_df.groupby("GEN_BUS").cumcount() + 1
# require an id present in the JSON (avoid the "not in JSON" branch)
json_gids = {d["id"] for d in ILLINOIS["devices"] if d.get("class") == "Genrou"}
candidates = gen_df[(gen_df["GEN_STATUS"] == 1) & (gen_df["PG"] > 10.0)].copy()
candidates["gid"] = (
    candidates["GEN_BUS"].astype(int).astype(str)
    + "_"
    + candidates["rank_at_bus"].astype(str)
)
candidates = candidates[candidates["gid"].isin(json_gids)]
row = candidates.iloc[0]
TARGET_GEN_BUS_4B = int(row["GEN_BUS"])
TARGET_GEN_RANK_4B = int(row["rank_at_bus"])
TARGET_GID_4B = row["gid"]
pg_base = float(row["PG"])
pg_new = pg_base + 100.0
print(
    f"gen {TARGET_GID_4B} (bus={TARGET_GEN_BUS_4B}, rank={TARGET_GEN_RANK_4B}): "
    f"PG {pg_base:.6f} MW -> {pg_new:.6f} MW"
)

M_4B = _edit_m_gen_col(
    BASE_SOLVED_M,
    WORK_DIR / "4b_gen_pg.m",
    gen_bus=TARGET_GEN_BUS_4B,
    rank=TARGET_GEN_RANK_4B,
    col_1based=2,
    new_value=pg_new,
)

case_4b, counts_4b = build_case_from_solved_m(
    base_case_json_path=str(ILLINOIS_JSON),
    m_path=str(M_4B),
    output_path=None,
    patch_bus_init=False,
    patch_gen_dispatch=True,
    patch_load=False,
    remove_offline_gens=False,
    strict=False,
    verbose=False,
)

p0_expected_delta = 100.0 / float(base_mp.baseMVA)
target = next(
    d
    for d in case_4b["devices"]
    if d.get("class") == "Genrou" and d.get("id") == TARGET_GID_4B
)
p0_ref = next(
    d
    for d in case_out["devices"]
    if d.get("class") == "Genrou" and d.get("id") == TARGET_GID_4B
)["params"]["p0"]
p0_got_delta = target["params"]["p0"] - p0_ref
print(f"expected dp0 = {p0_expected_delta:.9f}")
print(f"got      dp0 = {p0_got_delta:.9f}")
print(f"|error|      = {abs(p0_got_delta - p0_expected_delta):.3e}")

## 4c. offline-gen removal

Set `GEN_STATUS=0` on a gen that IS in the base JSON. Confirm that the
`{Genrou, Tgov1, SexsPti}` triplet is removed from `case["devices"]` and no
other device is affected.


In [ ]:
# Reuse the same target gen from 4b (guaranteed to be in the JSON).
M_4C = _edit_m_gen_col(
    BASE_SOLVED_M,
    WORK_DIR / "4c_gen_off.m",
    gen_bus=TARGET_GEN_BUS_4B,
    rank=TARGET_GEN_RANK_4B,
    col_1based=8,
    new_value=0.0,
)  # GEN_STATUS = 0

case_4c, counts_4c = build_case_from_solved_m(
    base_case_json_path=str(ILLINOIS_JSON),
    m_path=str(M_4C),
    output_path=None,
    patch_bus_init=False,
    patch_gen_dispatch=False,
    patch_load=False,
    remove_offline_gens=True,
    strict=False,
    verbose=True,
)
counts_4c

In [ ]:
# Structural check: exactly the {Genrou, Tgov1, SexsPti} triplet with id
# prefix TARGET_GID_4B should be gone. All other devices intact.
def id_class_bag(case_dict):
    return sorted((d.get("class"), d.get("id")) for d in case_dict.get("devices", []))


base_bag = set(id_class_bag(ILLINOIS))
out_bag = set(id_class_bag(case_4c))
removed = base_bag - out_bag
added = out_bag - base_bag

print(f"devices removed: {len(removed)}")
for cls, did in sorted(removed):
    print(f"  {cls:<10s} {did}")
print(f"devices added  : {len(added)}")
for cls, did in sorted(added):
    print(f"  {cls:<10s} {did}")

# Expected: 3 removed (Genrou + Tgov1 + SexsPti/Ieeet1 for TARGET_GID_4B), 0 added.
# All removed ids should start with TARGET_GID_4B.
prefix_ok = all(
    did == TARGET_GID_4B or did.startswith(TARGET_GID_4B + "_") for _, did in removed
)
print(f"all removed ids match prefix '{TARGET_GID_4B}': {prefix_ok}")

## 4d. LoadZIP single-bus patch (single-ZIP bus)

Edit `PD` and `QD` at a bus that has exactly one LoadZIP device. Verify only
that device's `Pnom`, `Qnom`, `Vnom` changed; `alphaI`, `alphaP` untouched;
all other LoadZIPs unchanged.


In [ ]:
# Pick a single-ZIP bus with PD > 0.
single_zip_buses = [b for b, devs in loadzip_by_bus.items() if len(devs) == 1]
bus_pd_series = base_mp.bus.set_index("BUS_I")["PD"].astype(float)
TARGET_BUS_4D = next(b for b in single_zip_buses if bus_pd_series.get(b, 0) > 1.0)
pd_base = float(bus_pd_series.loc[TARGET_BUS_4D])
qd_base = float(base_mp.bus.set_index("BUS_I").loc[TARGET_BUS_4D, "QD"])
pd_new = pd_base * 2.0
qd_new = qd_base * 2.0
print(
    f"bus {TARGET_BUS_4D}: PD {pd_base:.6f} -> {pd_new:.6f}, "
    f"QD {qd_base:.6f} -> {qd_new:.6f}"
)

M_4D_STEP1 = _edit_m_bus_col(
    BASE_SOLVED_M,
    WORK_DIR / "4d_load_pd.m",
    bus_i=TARGET_BUS_4D,
    col_1based=3,
    new_value=pd_new,
)
M_4D = _edit_m_bus_col(
    M_4D_STEP1,
    WORK_DIR / "4d_load_pdqd.m",
    bus_i=TARGET_BUS_4D,
    col_1based=4,
    new_value=qd_new,
)

case_4d, counts_4d = build_case_from_solved_m(
    base_case_json_path=str(ILLINOIS_JSON),
    m_path=str(M_4D),
    output_path=None,
    patch_bus_init=False,
    patch_gen_dispatch=False,
    patch_load=True,
    remove_offline_gens=False,
    strict=False,
    verbose=False,
)


# Check target LoadZIP.
def find_loadzip_at_bus(case_dict, bus_num):
    return [
        d
        for d in case_dict["devices"]
        if d.get("class") == "LoadZIP" and d.get("ports", {}).get("bus") == bus_num
    ]


target_lz = find_loadzip_at_bus(case_4d, TARGET_BUS_4D)[0]
ref_lz = find_loadzip_at_bus(case_out, TARGET_BUS_4D)[0]
baseMVA_here = float(base_mp.baseMVA)
print(
    f"target LoadZIP Pnom: ref={ref_lz['params']['Pnom']:.6f} -> "
    f"got={target_lz['params']['Pnom']:.6f} (expected {pd_new/baseMVA_here:.6f})"
)
print(
    f"target LoadZIP Qnom: ref={ref_lz['params']['Qnom']:.6f} -> "
    f"got={target_lz['params']['Qnom']:.6f} (expected {qd_new/baseMVA_here:.6f})"
)
print(
    f"alphaI unchanged: "
    f"{target_lz['params'].get('alphaI') == ref_lz['params'].get('alphaI')}"
)
print(
    f"alphaP unchanged: "
    f"{target_lz['params'].get('alphaP') == ref_lz['params'].get('alphaP')}"
)

# Count other LoadZIP changes vs case_out (should be zero).
n_other_lz_changed = 0
for d_new, d_ref in zip(
    [d for d in case_4d["devices"] if d.get("class") == "LoadZIP"],
    [d for d in case_out["devices"] if d.get("class") == "LoadZIP"],
):
    if d_new.get("ports", {}).get("bus") == TARGET_BUS_4D:
        continue
    for k in ("Pnom", "Qnom", "Vnom"):
        if abs(d_new["params"].get(k, 0) - d_ref["params"].get(k, 0)) > 1e-12:
            n_other_lz_changed += 1
            break
print(f"other LoadZIPs changed: {n_other_lz_changed} (expected 0)")

## 4e. LoadZIP multi-ZIP proportional-scaling exactness

Pick a bus with 2 LoadZIP devices; change `PD` in the `.m` by a factor of 2.
Verify `sum(Pnom_out) == PD_new/baseMVA` and the individual-device
`Pnom[0]/Pnom[1]` ratio is preserved to floating-point.


In [ ]:
TARGET_BUS_4E = next(b for b in multi_zip_buses if bus_pd_series.get(b, 0) > 1.0)
devs_at_bus = loadzip_by_bus[TARGET_BUS_4E]
pd_base = float(bus_pd_series.loc[TARGET_BUS_4E])
pd_new = pd_base * 2.0
print(
    f"multi-ZIP bus {TARGET_BUS_4E}: {len(devs_at_bus)} devices, "
    f"PD {pd_base:.6f} -> {pd_new:.6f}"
)

# Base per-device Pnom ratios in the reference (case_out post-identity-round-trip).
ref_lzs = find_loadzip_at_bus(case_out, TARGET_BUS_4E)
ref_pnoms = [d["params"]["Pnom"] for d in ref_lzs]
ref_ratio = (
    ref_pnoms[0] / ref_pnoms[1] if len(ref_pnoms) == 2 and ref_pnoms[1] != 0 else np.nan
)
print(f"ref Pnoms: {ref_pnoms}, ratio Pnom[0]/Pnom[1] = {ref_ratio:.9f}")

M_4E = _edit_m_bus_col(
    BASE_SOLVED_M,
    WORK_DIR / "4e_multizip_pd.m",
    bus_i=TARGET_BUS_4E,
    col_1based=3,
    new_value=pd_new,
)

case_4e, counts_4e = build_case_from_solved_m(
    base_case_json_path=str(ILLINOIS_JSON),
    m_path=str(M_4E),
    output_path=None,
    patch_bus_init=False,
    patch_gen_dispatch=False,
    patch_load=True,
    remove_offline_gens=False,
    strict=False,
    verbose=False,
)

out_lzs = find_loadzip_at_bus(case_4e, TARGET_BUS_4E)
out_pnoms = [d["params"]["Pnom"] for d in out_lzs]
out_ratio = (
    out_pnoms[0] / out_pnoms[1] if len(out_pnoms) == 2 and out_pnoms[1] != 0 else np.nan
)
sum_pnom = sum(out_pnoms)
expected_sum = pd_new / float(base_mp.baseMVA)
print(f"out Pnoms: {out_pnoms}")
print(f"sum(Pnom_out)   = {sum_pnom:.9f}")
print(f"expected sum    = {expected_sum:.9f}")
print(f"|error|          = {abs(sum_pnom - expected_sum):.3e}")
print(f"out ratio        = {out_ratio:.9f}")
print(f"|ratio delta|    = {abs(out_ratio - ref_ratio):.3e}")

## 4f. shunt / capacitor exclusion

The bus 15 LoadZIP has `Pnom=0, Qnom<0` (reactive-compensation capacitor per
`illinois.md`). Even after LoadZIP patching, its `Pnom` and `Qnom` must stay
put; only `Vnom` should refresh to the bus voltage from the solved `.m`.


In [ ]:
SHUNT_BUS = 15

# Confirm bus 15 has exactly one shunt-like LoadZIP in the base JSON.
shunt_lzs = [
    d
    for d in base_case["devices"]
    if d.get("class") == "LoadZIP" and d.get("ports", {}).get("bus") == SHUNT_BUS
]
assert (
    len(shunt_lzs) == 1
), f"expected 1 LoadZIP at bus {SHUNT_BUS}, got {len(shunt_lzs)}"
s = shunt_lzs[0]
print(
    f"base shunt LoadZIP id={s['id']}, "
    f"Pnom={s['params']['Pnom']}, Qnom={s['params']['Qnom']}, "
    f"Vnom={s['params']['Vnom']:.6f}"
)
assert (
    s["params"]["Pnom"] == 0.0 and s["params"]["Qnom"] < 0
), "bus 15 LoadZIP is not a shunt (Pnom!=0 or Qnom>=0)"

# Run builder with patch_load=True on the BASE_SOLVED_M; no edits needed.
case_4f, counts_4f = build_case_from_solved_m(
    base_case_json_path=str(ILLINOIS_JSON),
    m_path=str(BASE_SOLVED_M),
    output_path=None,
    patch_bus_init=False,
    patch_gen_dispatch=False,
    patch_load=True,
    remove_offline_gens=False,
    strict=False,
    verbose=False,
)

out_shunt = [
    d
    for d in case_4f["devices"]
    if d.get("class") == "LoadZIP" and d.get("ports", {}).get("bus") == SHUNT_BUS
][0]
print(
    f"out  shunt LoadZIP Pnom={out_shunt['params']['Pnom']}, "
    f"Qnom={out_shunt['params']['Qnom']}, "
    f"Vnom={out_shunt['params']['Vnom']:.6f}"
)

print(f"Pnom unchanged: {out_shunt['params']['Pnom'] == s['params']['Pnom']}")
print(f"Qnom unchanged: {out_shunt['params']['Qnom'] == s['params']['Qnom']}")

# Vnom should reflect the bus voltage from the solved .m.
from pf_utils import read_matpower_case as _rmc2

_mp2 = _rmc2(str(BASE_SOLVED_M))
vm_bus15 = float(_mp2.bus.set_index("BUS_I").loc[SHUNT_BUS, "VM"])
print(f"VM at bus {SHUNT_BUS}: {vm_bus15:.6f}")
print(f"Vnom refreshed: {abs(out_shunt['params']['Vnom'] - vm_bus15) < 1e-12}")
print(f"shunt_skipped count: {counts_4f['shunt_skipped']}")

## 4g. zero-load bus

Force `PD = QD = 0` on a single-ZIP bus. The builder should set
`Pnom = Qnom = 0` on the LoadZIP and refresh `Vnom` from the bus voltage.


In [ ]:
TARGET_BUS_4G = next(
    b for b in single_zip_buses if bus_pd_series.get(b, 0) > 1.0 and b != TARGET_BUS_4D
)

M_4G_STEP1 = _edit_m_bus_col(
    BASE_SOLVED_M,
    WORK_DIR / "4g_zero_pd.m",
    bus_i=TARGET_BUS_4G,
    col_1based=3,
    new_value=0.0,
)
M_4G = _edit_m_bus_col(
    M_4G_STEP1,
    WORK_DIR / "4g_zero_pdqd.m",
    bus_i=TARGET_BUS_4G,
    col_1based=4,
    new_value=0.0,
)

case_4g, counts_4g = build_case_from_solved_m(
    base_case_json_path=str(ILLINOIS_JSON),
    m_path=str(M_4G),
    output_path=None,
    patch_bus_init=False,
    patch_gen_dispatch=False,
    patch_load=True,
    remove_offline_gens=False,
    strict=False,
    verbose=False,
)

target_lz_4g = find_loadzip_at_bus(case_4g, TARGET_BUS_4G)[0]
print(f"bus {TARGET_BUS_4G} zero-load LoadZIP:")
print(f"  Pnom = {target_lz_4g['params']['Pnom']} (expected 0.0)")
print(f"  Qnom = {target_lz_4g['params']['Qnom']} (expected 0.0)")
print(f"  Vnom = {target_lz_4g['params']['Vnom']:.6f}")

from pf_utils import read_matpower_case as _rmc3

_mp3 = _rmc3(str(BASE_SOLVED_M))
vm_4g = float(_mp3.bus.set_index("BUS_I").loc[TARGET_BUS_4G, "VM"])
print(f"  VM   = {vm_4g:.6f}")
print(f"Vnom refreshed to VM: {abs(target_lz_4g['params']['Vnom'] - vm_4g) < 1e-12}")

## 5. End-to-end: 0.8× load perturbation

Scale all loads to 80% using `pf_utils.make_load_scale_m`, re-solve with PM.jl,
build `case.json`, and verify:

- `sum(LoadZIP.Pnom_out)` is within tolerance of `0.8 × sum(LoadZIP.Pnom_base)`.
- All non-shunt LoadZIPs have updated `Pnom`, `Qnom`, `Vnom`.


In [ ]:
from pf_utils import make_load_scale_m

SCALE = 0.8
M_SCALED_SRC = WORK_DIR / "5_scale08_src.m"
M_SCALED_SOLVED = WORK_DIR / "5_scale08_solved.m"

make_load_scale_m(str(BASECASE_M), str(M_SCALED_SRC), SCALE)
print(f"scaled .m written: {M_SCALED_SRC}")

run_pm_solve_out(
    jl_script=str(Path(PM_PROJECT_DIR) / "pm_solve.jl"),
    input_m=str(M_SCALED_SRC),
    output_m=str(M_SCALED_SOLVED),
    julia_bin=JULIA_BIN,
    project_dir=PM_PROJECT_DIR,
)
print(f"solved .m written: {M_SCALED_SOLVED}")

In [ ]:
OUT_JSON_5 = WORK_DIR / "5_case_scale08.json"

case_5, counts_5 = build_case_from_solved_m(
    base_case_json_path=str(ILLINOIS_JSON),
    m_path=str(M_SCALED_SOLVED),
    output_path=str(OUT_JSON_5),
    patch_bus_init=True,
    patch_gen_dispatch=True,
    patch_load=True,
    remove_offline_gens=True,
    strict=True,
    verbose=True,
)
print(f"\ncounts: {counts_5}")

# Verify global load sum: sum(Pnom out) ≈ 0.8 × sum(Pnom base).
base_lz_pnom = [
    d["params"]["Pnom"]
    for d in base_case["devices"]
    if d.get("class") == "LoadZIP"
    and not (d["params"]["Pnom"] == 0.0 and d["params"]["Qnom"] < 0)
]
out_lz_pnom = [
    d["params"]["Pnom"]
    for d in case_5["devices"]
    if d.get("class") == "LoadZIP"
    and not (d["params"]["Pnom"] == 0.0 and d["params"]["Qnom"] < 0)
]

base_sum = sum(base_lz_pnom)
out_sum = sum(out_lz_pnom)
print(f"\nbase sum(Pnom non-shunt) = {base_sum:.6f}")
print(f"out  sum(Pnom non-shunt) = {out_sum:.6f}")
print(f"ratio out/base           = {out_sum/base_sum:.6f} (expected {SCALE:.6f})")
print(f"|ratio - {SCALE}|          = {abs(out_sum/base_sum - SCALE):.3e}")

## 6. Findings summary

All assertions below are filled in after notebook execution.

| Test | Result | Notes |
|------|--------|-------|
| 3.3 structural check | [TODO] | device-class counts unchanged |
| 3.4 bus Vr/Vi max residual | [TODO] | should be < 1e-9 |
| 3.5 Genrou p0/q0 max residual | [TODO] | should be < 1e-9 |
| 3.6 LoadZIP fraction invariant max error | [TODO] | should be < 1e-12 |
| 3.7 LoadZIP aggregate vs PD max error | [TODO] | should be < 1e-9 |
| 4a bus init (Vr/Vi from VM/VA) | [TODO] | exact to floating-point |
| 4b Genrou dispatch (dp0) | [TODO] | exact to floating-point |
| 4c offline-gen triplet removed | [TODO] | set-diff count should be 3 |
| 4d single-ZIP patch (Pnom/Qnom) | [TODO] | exact; other LoadZIPs unchanged |
| 4e multi-ZIP ratio preserved | [TODO] | |delta ratio| < 1e-12 |
| 4f shunt Pnom/Qnom frozen, Vnom refreshed | [TODO] | shunt_skipped >= 1 |
| 4g zero-load Pnom=Qnom=0 | [TODO] | Vnom = bus VM |
| 5 end-to-end 0.8x: ratio out/base | [TODO] | should be 0.8 ± 1e-6 |

### Known limitations

1. Cannot add a new generator (online in `.m` but absent in base JSON); `strict=True` raises `ValueError`. Requires rebuild.
2. LoadZ models not supported; only LoadZIP.
3. ZIP parameter mix (`alphaI`, `alphaP`) preserved unchanged from base JSON.
4. Multi-shunt-per-bus not tested (bus 15 has exactly one shunt-like device).
